## we're starting with installing library thats gonna be used within codes.

In [1]:
!pip install pandas faker numpy sqlalchemy psycopg2-binary scikit-learn
#pip = used to install libraries to import codes
#pandas = used for data manipulation, reading csvs to hold and clean tablular data
#faker = it generate relaistics fake data
#numpy = it is used for numerical computations
#sqlalchemy = it lets python talk to databases using python code instead of sql
#psyycopg2 = its an actual driver which tells python connect to plsql databases
#scikit-learn = ml classification

## here we import all the libraries we gonna use in it

In [2]:
from faker import Faker
import pandas as pd
import numpy as np
import random
from datetime import timedelta

## here we're creating actual customer table by creating dictionary and then convert it into tabular form

In [3]:
fake = Faker('en_IN')
random.seed(42)
np.random.seed(42)

N_CUSTOMERS = 2000
N_ACCOUNTS = 2500

customers = []
for i in range(1, N_CUSTOMERS + 1):
 customers.append({
  "customer_id": i,
  "full_name": fake.name(),
  "dob": fake.date_of_birth(minimum_age = 18, maximum_age = 75),
  "city": fake.city(),
  "kyc_risk_rating": random.choices(["Low", "Medium", "High"], weights = [0.75, 0.20, 0.05])[0],
  "account_open_date": fake.date_between(start_date = "-5y", end_date = "-30d")
 })
customers_df  =pd.DataFrame(customers)


In [4]:
customers_df.head()

,customer_id,full_name,dob,city,kyc_risk_rating,account_open_date
0,1,Shaurya Jaggi,1993-10-16,Solapur,Low,2023-07-17
1,2,Kashvi Gole,1966-08-17,Mira-Bhayandar,Low,2023-09-23
2,3,Dalaja Biswas,1980-10-31,Jamalpur,Low,2025-02-05
3,4,Jonathan Kumer,1988-08-29,Dewas,Low,2022-10-15
4,5,Laban Kota,1968-07-12,Raichur,Low,2023-11-18


## we're creating account table

In [5]:
N_ACCOUNTS = 2500
accounts = []
for i in range(1, N_ACCOUNTS + 1):
 cust = random.choice(customers)
 accounts.append({
  "account_id": i,
  "customer_id": cust["customer_id"],
  "account_type": random.choices(["Savings", "Current"], weights = [0.85, 0.15])[0],
  "branch": fake.city(),
  "opened_date": cust["account_open_date"]
 })
accounts_df = pd.DataFrame(accounts)

In [6]:
accounts_df.head()

,account_id,customer_id,account_type,branch,opened_date
0,1,294,Current,Alwar,2025-11-07
1,2,1068,Savings,Rajahmundry,2026-03-03
2,3,616,Savings,Vellore,2023-12-25
3,4,1141,Current,Bettiah,2026-06-29
4,5,1802,Savings,Ongole,2023-10-19


## here we're creating normal transaction traffic, this is the backgrounf noise our fraud pattern will hide inside here.

In [7]:
def random_timestamp(start, end):
 delta = end - start
 return start + timedelta(seconds = random.randint(0, int(delta.total_seconds())))
start_date = pd.Timestamp("2024-01-01")
end_date = pd.Timestamp("2025-12-31")

transactions = []
txn_id = 1
account_ids = accounts_df["account_id"].tolist()

for _ in range(60000):
 from_acc, to_acc = random.sample(account_ids, 2)
 transactions.append({
  "txn_id": txn_id,
  "from_account_id": from_acc,
  "to_account_id": to_acc,
  "amount": round(np.random.lognormal(mean=8, sigma=1.2), 2),
  "txn_timestamp": random_timestamp(start_date, end_date),
  "txn_type": random.choice(["NEFT", "IMPS", "UPI", "RIGS"]),
  "channel": random.choice(["Mobile", "Net Banking", "Branch"]),
  "location": fake.city()
 })
txn_id += 1
transactions_df = pd.DataFrame(transactions)

In [8]:
len(transactions)

60000

In [9]:
transactions_df.head()

,txn_id,from_account_id,to_account_id,amount,txn_timestamp,txn_type,channel,location
0,1,2377,860,5410.28,2025-08-08 00:10:17,IMPS,Net Banking,Bettiah
1,1,1445,1096,2525.22,2025-01-26 03:09:00,RIGS,Net Banking,Panipat
2,1,224,260,6484.86,2024-10-16 13:45:09,UPI,Net Banking,Pune
3,1,1992,2054,18539.07,2025-02-13 02:04:30,NEFT,Net Banking,Bally
4,1,2179,2183,2250.74,2025-11-29 14:59:08,UPI,Net Banking,Shimoga


## here we're create fraud pattern 1

In [10]:
def inject_structuring(account_ids, txn_id_start):
    injected = []
    txn_id = txn_id_start
    structuring_accounts = random.sample(account_ids, 15)
    for acc in structuring_accounts:
        base_time = random_timestamp(start_date, end_date)
        counterpart = random.choice(account_ids)
        for i in range(random.randint(3, 5)):
            injected.append({
                "txn_id": txn_id,
                "from_account_id": acc,
                "to_account_id": counterpart,
                "amount": round(random.uniform(180000, 199000), 2),
                "txn_timestamp": base_time + timedelta(hours = i*3),
                "txn_type": "IMPS",
                "channel": "Mobile",
                "location": fake.city()
            })
            txn_id += 1
    return injected, txn_id

In [11]:
structuring_txns, txn_id = inject_structuring(account_ids, txn_id)
transactions.extend(structuring_txns)

## 2nd fraud pattern here

In [12]:
def inject_round_tripping(account_ids, txn_id_start):
    injected = []
    txn_id = txn_id_start
    for _ in range(10):
        a, b, c = random.sample(account_ids, 3)
        amount = round(random.uniform(300000, 900000), 2)
        base_time = random_timestamp(start_date, end_date)
        for step, (frm, to) in enumerate([(a, b), (b, c), (c, a)]):
            injected.append({
                "txn_id": txn_id,
                "from_account_id": frm,
                "to_account_id": to,
                "amount": amount * random.uniform(0.95, 1.0),   
                "txn_timestamp": base_time + timedelta(days = step * 2),   
                "txn_type": "RTGS",   
                "channel": "Net Banking",
                "location": fake.city(),
            })
            txn_id += 1
    return injected, txn_id

In [13]:
roundtrip_txns, txn_id = inject_round_tripping(account_ids, txn_id)
transactions.extend(roundtrip_txns)

## 3rd fraud pattern here

In [14]:
def inject_mule_pattern(account_ids, txn_id_start):
    injected = []
    txn_id = txn_id_start
    mule_accounts = random.sample(account_ids, 10)
    for acc in mule_accounts:
        base_time = random_timestamp(start_date, end_date)
        total = 0
        for i in range(random.randint(8, 12)):
            depositor = random.choice(account_ids)
            amt = round(random.uniform(5000, 15000), 2)
            total += amt
            injected.append({
                "txn_id": txn_id,
                "from_account_id": depositor,
                "to_account_id": acc,
                "amount": amt,   
                "txn_timestamp": base_time + timedelta(hours = i),   
                "txn_type": "UPI",   
                "channel": "Mobile",
                "location": fake.city(),
            })
            #one big withdrawal shortly after
            txn_id += 1
        injected.append({
            "txn_id": txn_id,
            "from_account_id": acc,
            "to_account_id": random.choice(account_ids),
            "amount": round(total * 0.9, 2),
            "txn_timestamp": base_time + timedelta(days = 1),
            "txn_type": "RTGS",
            "channel": "Net Banking",
            "location": fake.city()
        })
        txn_id += 1
    return injected, txn_id

In [15]:
mule_txns, txn_id = inject_mule_pattern(account_ids, txn_id)
transactions.extend(mule_txns)

transactions_df = pd.DataFrame(transactions)
print(len(transactions_df))

60202


## loading everything into postgresql

In [16]:
from sqlalchemy import create_engine

username = "postgres"
password = "aml123"
host = "localhost"
port = "5432"
database = "aml_monitoring"

engine = create_engine(f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}")

In [17]:
from sqlalchemy import text

with engine.begin() as conn:
    # drop views first since they depend on the tables
    conn.execute(text("DROP VIEW IF EXISTS account_risk_scorecard"))
    conn.execute(text("DROP VIEW IF EXISTS rule1_structuring"))
    conn.execute(text("DROP VIEW IF EXISTS rule2_velocity"))
    conn.execute(text("DROP VIEW IF EXISTS rule3_roundtripping"))
    conn.execute(text("DROP VIEW IF EXISTS rule4_layering"))
    conn.execute(text("DROP VIEW IF EXISTS rule5_mule"))
    conn.execute(text("DROP TABLE IF EXISTS transactions"))
    conn.execute(text("DROP TABLE IF EXISTS accounts"))
    conn.execute(text("DROP TABLE IF EXISTS customers"))

customers_df.to_sql("customers", engine, if_exists="replace", index=False)
accounts_df.to_sql("accounts", engine, if_exists="replace", index=False)
transactions_df.to_sql("transactions", engine, if_exists="replace", index=False)

print("All tables loaded successfully")

All tables loaded successfully


## here we're debugging the errors that we get while working in sql

In [18]:
# this is to check that answer should be 0
transactions_df["txn_id"].duplicated().sum()

np.int64(59999)

In [19]:
len(transactions_df)

60202

In [20]:
## this is to fix the problem that i get when i tried to run ALTER TABLE transactions ADD PRIMARY KEY (txn_id) in Postgres, 
#and it failed and shows an error saying txn_id = 1 was duplicated. 
#A primary key requires every value to be unique but somewhere in the notebook's editing history, 
#a cell had gotten re-run out of order, which caused the manually-incremented txn_id counter to reset 
#and overlap with IDs that were already used.
transactions_df = transactions_df.reset_index(drop=True)
transactions_df["txn_id"] = transactions_df.index + 1

In [21]:
transactions_df["txn_id"].duplicated().sum()

np.int64(0)

In [22]:
transactions_df.to_sql("transactions", engine, if_exists="replace", index=False)
print("Reloaded successfully")

Reloaded successfully


In [23]:
#here in sql we only get the some rows, to get all 30 here we redo the code
transactions_df[
    (transactions_df["txn_type"] == "RTGS") & 
    (transactions_df["channel"] == "Net Banking") &
    (transactions_df["amount"].between(300000, 900000))
].shape[0]

30

In [24]:
print(len(structuring_txns))

64


In [25]:
print(len(roundtrip_txns))

30


In [26]:
print(len(mule_txns))

108
